# 🔍 Vector Store Viewer - Diagnostic Dashboard

A comprehensive diagnostic tool for exploring and debugging vector databases across all projects.

## Features:
- 🗂️ **Table Discovery**: Auto-detect all LanceDB tables
- 🔄 **Interactive Switching**: Easy toggle between different vector stores
- 📊 **Schema Analysis**: Deep dive into table structures and embeddings
- 🧪 **Vector Diagnostics**: Analyze dense vs multi-vector embeddings
- 📈 **Visualizations**: Data distributions and embedding projections
- 🔍 **Search Testing**: Quick query testing and performance comparison
- ✅ **Health Checks**: Data integrity validation

## Supported Projects:
- **ColBERT**: Dense vs ColBERT embeddings
- **Reasoning Retrievers**: Multi-step reasoning embeddings
- **Future Projects**: Extensible for any LanceDB vector store

## 1. Setup & Configuration

In [1]:
# Setup environment
import sys
sys.path.append('..')  # Go up to project root
from setup import *

# Additional imports for diagnostics
import lancedb
from pathlib import Path
import json
from typing import List, Dict, Any, Optional, Tuple
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Interactive widgets
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output, HTML
    WIDGETS_AVAILABLE = True
except ImportError:
    print("⚠️  ipywidgets not available. Install with: pip install ipywidgets")
    WIDGETS_AVAILABLE = False

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('default')
sns.set_palette("husl")

print(f"🔧 Vector Store Viewer initialized")
print(f"📂 Working directory: {os.getcwd()}")
print(f"🎯 Project root: {os.getenv('PROJECT_ROOT')}")
print(f"💾 Vector store directory: {os.getenv('VECTOR_STORE_DIR')}")
print(f"🎛️  Interactive widgets: {'✅' if WIDGETS_AVAILABLE else '❌'}")

✅ Loaded environment from: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/notebooks/../.env
📂 Working directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
🎯 Project root: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
📊 Data directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/data
🔧 Device: mps
📁 Environment variables loaded: 82
🔧 Vector Store Viewer initialized
📂 Working directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
🎯 Project root: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation
💾 Vector store directory: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/vector_store
🎛️  Interactive widgets: ✅


## 2. Table Discovery & Connection

In [2]:
class VectorStoreExplorer:
    """Comprehensive vector store diagnostic tool"""
    
    def __init__(self, vector_store_path: str = None):
        self.vector_store_path = vector_store_path or os.getenv('VECTOR_STORE_DIR', './vector_store')
        self.db = None
        self.current_table = None
        self.current_table_name = None
        self.tables_info = {}
        
        self.connect()
        
    def connect(self):
        """Connect to LanceDB and discover tables"""
        try:
            self.db = lancedb.connect(self.vector_store_path)
            self.discover_tables()
            print(f"✅ Connected to LanceDB at: {self.vector_store_path}")
            print(f"📊 Found {len(self.tables_info)} tables")
        except Exception as e:
            print(f"❌ Failed to connect to LanceDB: {e}")
            self.db = None
    
    def discover_tables(self):
        """Discover and analyze all tables in the database"""
        if not self.db:
            return
            
        self.tables_info = {}
        table_names = self.db.table_names()
        
        for table_name in table_names:
            try:
                table = self.db.open_table(table_name)
                schema = table.schema
                row_count = len(table)
                
                # Analyze schema for embedding columns
                embedding_cols = []
                for field in schema:
                    field_type = str(field.type)
                    if 'list' in field_type.lower() and 'float' in field_type.lower():
                        embedding_cols.append({
                            'name': field.name,
                            'type': field_type,
                            'is_multi_vector': 'list<list' in field_type.lower()
                        })
                
                self.tables_info[table_name] = {
                    'table': table,
                    'schema': schema,
                    'row_count': row_count,
                    'embedding_columns': embedding_cols,
                    'created_at': datetime.now().isoformat()
                }
                
            except Exception as e:
                print(f"⚠️  Could not analyze table {table_name}: {e}")
    
    def get_table_summary(self) -> pd.DataFrame:
        """Get summary of all discovered tables"""
        summaries = []
        
        for name, info in self.tables_info.items():
            embedding_info = []
            for col in info['embedding_columns']:
                emb_type = "Multi-Vector" if col['is_multi_vector'] else "Dense"
                embedding_info.append(f"{col['name']} ({emb_type})")
            
            summaries.append({
                'Table Name': name,
                'Rows': f"{info['row_count']:,}",
                'Columns': len(info['schema']),
                'Embedding Columns': ', '.join(embedding_info) if embedding_info else 'None',
                'Type': self._guess_table_type(name, info)
            })
        
        return pd.DataFrame(summaries)
    
    def _guess_table_type(self, name: str, info: dict) -> str:
        """Guess the table type based on name and schema"""
        name_lower = name.lower()
        
        if 'colbert' in name_lower:
            return 'ColBERT'
        elif 'dense' in name_lower:
            return 'Dense Embedding'
        elif 'reasoning' in name_lower:
            return 'Reasoning Retriever'
        elif any(col['is_multi_vector'] for col in info['embedding_columns']):
            return 'Multi-Vector'
        elif info['embedding_columns']:
            return 'Vector Store'
        else:
            return 'Unknown'

# Initialize explorer
explorer = VectorStoreExplorer()

if explorer.db:
    print("\n📋 Discovered Tables:")
    summary_df = explorer.get_table_summary()
    if not summary_df.empty:
        display(summary_df)
    else:
        print("   No tables found in the database")
else:
    print("❌ Could not connect to database. Check vector store path.")

✅ Connected to LanceDB at: /Users/luvsuneja/Documents/repos/advanced-rag-experimentation/vector_store
📊 Found 1 tables

📋 Discovered Tables:


,Table Name,Rows,Columns,Embedding Columns,Type
0,dense_reviews,12,7,dense_embedding (Dense),Dense Embedding


## 3. Interactive Table Selector

In [3]:
def create_table_selector():
    """Create interactive table selector widget"""
    if not WIDGETS_AVAILABLE or not explorer.tables_info:
        print("Manual table selection (widgets not available or no tables found)")
        for i, name in enumerate(explorer.tables_info.keys()):
            print(f"  {i+1}. {name}")
        return
    
    # Create dropdown widget
    table_options = list(explorer.tables_info.keys())
    table_dropdown = widgets.Dropdown(
        options=table_options,
        value=table_options[0] if table_options else None,
        description='Select Table:',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='400px')
    )
    
    output_area = widgets.Output()
    
    def on_table_change(change):
        """Handle table selection change"""
        with output_area:
            clear_output(wait=True)
            if change['new']:
                explorer.current_table_name = change['new']
                explorer.current_table = explorer.tables_info[change['new']]['table']
                show_table_info(change['new'])
    
    table_dropdown.observe(on_table_change, names='value')
    
    # Initialize with first table
    if table_options:
        explorer.current_table_name = table_options[0]
        explorer.current_table = explorer.tables_info[table_options[0]]['table']
    
    display(table_dropdown)
    display(output_area)
    
    # Show initial table info
    if table_options:
        with output_area:
            show_table_info(table_options[0])

def show_table_info(table_name: str):
    """Display detailed information about selected table"""
    info = explorer.tables_info[table_name]
    
    print(f"🔍 Table: {table_name}")
    print(f"📊 Rows: {info['row_count']:,}")
    print(f"📋 Columns: {len(info['schema'])}")
    print(f"🕒 Analyzed: {info['created_at'][:19]}")
    
    print(f"\n📐 Schema:")
    for field in info['schema']:
        field_type = str(field.type)
        is_embedding = any(col['name'] == field.name for col in info['embedding_columns'])
        marker = "🎯" if is_embedding else "📝"
        print(f"   {marker} {field.name}: {field_type}")
    
    if info['embedding_columns']:
        print(f"\n🔢 Embedding Columns:")
        for col in info['embedding_columns']:
            emb_type = "Multi-Vector" if col['is_multi_vector'] else "Dense"
            print(f"   • {col['name']}: {emb_type}")

# Create the selector
create_table_selector()

Dropdown(description='Select Table:', layout=Layout(width='400px'), options=('dense_reviews',), style=Descript…

Output()

## 4. Data Explorer Dashboard

In [4]:
def explore_current_table(num_samples: int = 3):
    """Explore the currently selected table"""
    if not explorer.current_table or not explorer.current_table_name:
        print("❌ No table selected")
        return
    
    print(f"🔍 Exploring: {explorer.current_table_name}")
    print("=" * 60)
    
    # Get sample data
    sample_df = explorer.current_table.head(num_samples).to_pandas()
    info = explorer.tables_info[explorer.current_table_name]
    
    print(f"\n📊 Basic Statistics:")
    print(f"   Total rows: {info['row_count']:,}")
    print(f"   Columns: {len(info['schema'])}")
    print(f"   Memory usage: ~{sample_df.memory_usage(deep=True).sum() * info['row_count'] / len(sample_df) / 1024 / 1024:.1f} MB (estimated)")
    
    print(f"\n📋 Column Analysis:")
    for col in sample_df.columns:
        col_data = sample_df[col]
        
        # Check if it's an embedding column
        is_embedding = any(emb_col['name'] == col for emb_col in info['embedding_columns'])
        
        if is_embedding:
            # Analyze embedding column
            first_embedding = col_data.iloc[0]
            if isinstance(first_embedding, list):
                if isinstance(first_embedding[0], list):  # Multi-vector
                    num_vectors = len(first_embedding)
                    vector_dim = len(first_embedding[0]) if first_embedding else 0
                    print(f"   🎯 {col}: {num_vectors} vectors × {vector_dim}D (Multi-vector)")
                else:  # Dense vector
                    vector_dim = len(first_embedding)
                    print(f"   🎯 {col}: {vector_dim}D (Dense vector)")
            else:
                print(f"   🎯 {col}: Unknown embedding format")
        else:
            # Regular column
            dtype = col_data.dtype
            unique_count = col_data.nunique()
            print(f"   📝 {col}: {dtype} ({unique_count} unique values)")
    
    print(f"\n📄 Sample Data:")
    for idx, row in sample_df.iterrows():
        print(f"\n   Row {idx + 1}:")
        for col, value in row.items():
            is_embedding = any(emb_col['name'] == col for emb_col in info['embedding_columns'])
            
            if is_embedding:
                if isinstance(value, list) and value:
                    if isinstance(value[0], list):  # Multi-vector
                        print(f"     {col}: [{len(value)} vectors] First: [{', '.join(f'{x:.3f}' for x in value[0][:3])}...]")
                    else:  # Dense vector
                        print(f"     {col}: [{', '.join(f'{x:.3f}' for x in value[:3])}...] ({len(value)}D)")
                else:
                    print(f"     {col}: {value}")
            else:
                # Truncate long text
                if isinstance(value, str) and len(value) > 100:
                    print(f"     {col}: {value[:100]}...")
                else:
                    print(f"     {col}: {value}")

# Explore current table
explore_current_table()

🔍 Exploring: dense_reviews

📊 Basic Statistics:
   Total rows: 12
   Columns: 7
   Memory usage: ~0.0 MB (estimated)

📋 Column Analysis:
   📝 id: int64 (3 unique values)
   📝 restaurant: object (3 unique values)
   📝 review: object (3 unique values)
   📝 reviewer: object (3 unique values)
   📝 rating: int64 (2 unique values)
   📝 text: object (3 unique values)
   🎯 dense_embedding: Unknown embedding format

📄 Sample Data:

   Row 1:
     id: 1
     restaurant: Mario's Bistro
     review: OMG this little Italian place is a hidden gem! 😍 Went there last night with my boyfriend and we sat ...
     reviewer: Sarah M.
     rating: 5
     text: Mario's Bistro: OMG this little Italian place is a hidden gem! 😍 Went there last night with my boyfr...
     dense_embedding: [-3.51779796e-02  4.94697541e-02  3.36786956e-02  8.62713382e-02
 -8.94316807e-02 -4.23065247e-03  5.37196174e-02 -2.77091675e-02
 -1.37019325e-02 -1.37534335e-01 -3.05618159e-02 -2.01999675e-02
  1.21028051e-02 -3.24191973e-02

## 5. Vector Diagnostics

In [5]:
def analyze_embeddings():
    """Detailed analysis of embedding columns"""
    if not explorer.current_table or not explorer.current_table_name:
        print("❌ No table selected")
        return
    
    info = explorer.tables_info[explorer.current_table_name]
    
    if not info['embedding_columns']:
        print("❌ No embedding columns found in current table")
        return
    
    print(f"🎯 Embedding Analysis: {explorer.current_table_name}")
    print("=" * 60)
    
    # Get sample data for analysis
    sample_df = explorer.current_table.to_pandas()  # Get all data for better analysis
    
    for emb_col in info['embedding_columns']:
        col_name = emb_col['name']
        is_multi_vector = emb_col['is_multi_vector']
        
        print(f"\n📊 {col_name} ({'Multi-Vector' if is_multi_vector else 'Dense Vector'}):")
        
        embeddings = sample_df[col_name].tolist()
        
        if is_multi_vector:
            # Multi-vector analysis (ColBERT style)
            vector_counts = [len(emb) if emb else 0 for emb in embeddings]
            dimensions = [len(emb[0]) if emb and emb[0] else 0 for emb in embeddings]
            
            print(f"   📈 Vector counts per document:")
            print(f"      Min: {min(vector_counts)} | Max: {max(vector_counts)} | Avg: {np.mean(vector_counts):.1f}")
            print(f"   📐 Dimensions: {max(dimensions) if dimensions else 0}D")
            print(f"   💾 Total vectors: {sum(vector_counts):,}")
            print(f"   📦 Storage estimate: {sum(vector_counts) * max(dimensions) * 4 / 1024 / 1024:.1f} MB")
            
            # Distribution plot
            if len(vector_counts) > 1:
                plt.figure(figsize=(10, 4))
                
                plt.subplot(1, 2, 1)
                plt.hist(vector_counts, bins=min(20, len(set(vector_counts))), alpha=0.7)
                plt.xlabel('Vectors per Document')
                plt.ylabel('Frequency')
                plt.title(f'{col_name}: Vector Count Distribution')
                plt.grid(True, alpha=0.3)
                
                plt.subplot(1, 2, 2)
                plt.plot(vector_counts, 'o-', alpha=0.7, markersize=4)
                plt.xlabel('Document Index')
                plt.ylabel('Number of Vectors')
                plt.title(f'{col_name}: Vectors per Document')
                plt.grid(True, alpha=0.3)
                
                plt.tight_layout()
                plt.show()
        
        else:
            # Dense vector analysis
            dimensions = [len(emb) if emb else 0 for emb in embeddings]
            
            print(f"   📐 Dimensions: {max(dimensions) if dimensions else 0}D")
            print(f"   📦 Total embeddings: {len(embeddings):,}")
            print(f"   💾 Storage estimate: {len(embeddings) * max(dimensions) * 4 / 1024 / 1024:.1f} MB")
            
            # Check dimension consistency
            if dimensions and len(set(dimensions)) > 1:
                print(f"   ⚠️  Inconsistent dimensions found: {set(dimensions)}")
            else:
                print(f"   ✅ Consistent dimensions across all vectors")

# Run embedding analysis
analyze_embeddings()

🎯 Embedding Analysis: dense_reviews

📊 dense_embedding (Dense Vector):


ValueError: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()

## 6. Search Testing Interface

In [6]:
def quick_search_test(query: str = "Italian restaurant", limit: int = 3):
    """Quick search test on current table"""
    if not explorer.current_table or not explorer.current_table_name:
        print("❌ No table selected")
        return
    
    info = explorer.tables_info[explorer.current_table_name]
    
    if not info['embedding_columns']:
        print("❌ No embedding columns found - cannot perform search")
        return
    
    print(f"🔍 Quick Search Test: '{query}'")
    print(f"📊 Table: {explorer.current_table_name}")
    print("=" * 60)
    
    # For demonstration, we'll show how to structure the search
    # Note: Actual search would require the embedding model to encode the query
    
    try:
        # Get sample results (top N rows as demonstration)
        results_df = explorer.current_table.head(limit).to_pandas()
        
        print(f"📋 Sample Results (Top {limit} rows):")
        for idx, row in results_df.iterrows():
            print(f"\n   Result {idx + 1}:")
            
            # Show non-embedding columns
            for col, value in row.items():
                is_embedding = any(emb_col['name'] == col for emb_col in info['embedding_columns'])
                
                if not is_embedding:
                    if isinstance(value, str) and len(value) > 100:
                        print(f"     {col}: {value[:100]}...")
                    else:
                        print(f"     {col}: {value}")
        
        print(f"\n💡 Note: This is a demonstration. For actual semantic search:")
        print(f"   1. Load the appropriate embedding model")
        print(f"   2. Encode the query: '{query}'")
        print(f"   3. Use LanceDB's .search() method with the query embedding")
        print(f"   4. For ColBERT tables, implement MaxSim operation")
        
    except Exception as e:
        print(f"❌ Search test failed: {e}")

def compare_tables_structure():
    """Compare structure of all available tables"""
    if len(explorer.tables_info) < 2:
        print("❌ Need at least 2 tables for comparison")
        return
    
    print("🔄 Table Structure Comparison")
    print("=" * 60)
    
    comparison_data = []
    
    for table_name, info in explorer.tables_info.items():
        embedding_info = []
        for col in info['embedding_columns']:
            emb_type = "Multi" if col['is_multi_vector'] else "Dense"
            embedding_info.append(f"{col['name']}({emb_type})")
        
        comparison_data.append({
            'Table': table_name,
            'Rows': info['row_count'],
            'Columns': len(info['schema']),
            'Embeddings': ', '.join(embedding_info) if embedding_info else 'None',
            'Type': explorer._guess_table_type(table_name, info)
        })
    
    comparison_df = pd.DataFrame(comparison_data)
    display(comparison_df)
    
    # Visual comparison
    if len(comparison_data) <= 10:  # Only plot if reasonable number of tables
        plt.figure(figsize=(12, 5))
        
        plt.subplot(1, 2, 1)
        table_names = [item['Table'] for item in comparison_data]
        row_counts = [item['Rows'] for item in comparison_data]
        
        plt.bar(range(len(table_names)), row_counts)
        plt.xlabel('Tables')
        plt.ylabel('Number of Rows')
        plt.title('Row Count Comparison')
        plt.xticks(range(len(table_names)), [name[:15] + '...' if len(name) > 15 else name for name in table_names], rotation=45)
        plt.grid(True, alpha=0.3)
        
        plt.subplot(1, 2, 2)
        column_counts = [item['Columns'] for item in comparison_data]
        
        plt.bar(range(len(table_names)), column_counts, color='orange')
        plt.xlabel('Tables')
        plt.ylabel('Number of Columns')
        plt.title('Column Count Comparison')
        plt.xticks(range(len(table_names)), [name[:15] + '...' if len(name) > 15 else name for name in table_names], rotation=45)
        plt.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

# Run search test
quick_search_test()

print("\n" + "="*60)

# Compare all tables
compare_tables_structure()

🔍 Quick Search Test: 'Italian restaurant'
📊 Table: dense_reviews
📋 Sample Results (Top 3 rows):

   Result 1:
     id: 1
     restaurant: Mario's Bistro
     review: OMG this little Italian place is a hidden gem! 😍 Went there last night with my boyfriend and we sat ...
     reviewer: Sarah M.
     rating: 5
     text: Mario's Bistro: OMG this little Italian place is a hidden gem! 😍 Went there last night with my boyfr...

   Result 2:
     id: 2
     restaurant: Sakura Sushi
     review: Finally found a sushi place that caters to vegetarians! They have this amazing tempura vegetable rol...
     reviewer: Alex T.
     rating: 4
     text: Sakura Sushi: Finally found a sushi place that caters to vegetarians! They have this amazing tempura...

   Result 3:
     id: 3
     restaurant: Code & Coffee
     review: As a freelance developer, I'm always hunting for good work spots and Code & Coffee is PERFECT. The w...
     reviewer: DevMike
     rating: 5
     text: Code & Coffee: As a freelance

## 7. Health Checks & Validation

In [7]:
def run_health_checks():
    """Run comprehensive health checks on current table"""
    if not explorer.current_table or not explorer.current_table_name:
        print("❌ No table selected")
        return
    
    print(f"🏥 Health Check: {explorer.current_table_name}")
    print("=" * 60)
    
    info = explorer.tables_info[explorer.current_table_name]
    health_score = 100
    issues = []
    warnings = []
    
    try:
        # Get data for analysis
        df = explorer.current_table.to_pandas()
        
        print(f"📊 Basic Checks:")
        
        # 1. Row count check
        if len(df) == 0:
            issues.append("Table is empty")
            health_score -= 50
        elif len(df) < 10:
            warnings.append(f"Very small dataset ({len(df)} rows)")
            health_score -= 10
        print(f"   ✅ Row count: {len(df):,}")
        
        # 2. Null value checks
        null_counts = df.isnull().sum()
        if null_counts.sum() > 0:
            null_cols = null_counts[null_counts > 0]
            for col, count in null_cols.items():
                pct = (count / len(df)) * 100
                if pct > 50:
                    issues.append(f"Column '{col}' has {pct:.1f}% null values")
                    health_score -= 20
                elif pct > 10:
                    warnings.append(f"Column '{col}' has {pct:.1f}% null values")
                    health_score -= 5
        print(f"   ✅ Null values: {null_counts.sum()} total")
        
        # 3. Embedding-specific checks
        print(f"\n🎯 Embedding Checks:")
        
        for emb_col in info['embedding_columns']:
            col_name = emb_col['name']
            is_multi_vector = emb_col['is_multi_vector']
            
            print(f"\n   📋 {col_name} ({'Multi-Vector' if is_multi_vector else 'Dense'}):")
            
            embeddings = df[col_name].tolist()
            
            # Check for empty embeddings
            empty_count = sum(1 for emb in embeddings if not emb or len(emb) == 0)
            if empty_count > 0:
                pct = (empty_count / len(embeddings)) * 100
                if pct > 10:
                    issues.append(f"Column '{col_name}' has {pct:.1f}% empty embeddings")
                    health_score -= 15
                else:
                    warnings.append(f"Column '{col_name}' has {empty_count} empty embeddings")
                    health_score -= 5
            print(f"     ✅ Empty embeddings: {empty_count}/{len(embeddings)}")
            
            if is_multi_vector:
                # Multi-vector specific checks
                vector_counts = [len(emb) if emb else 0 for emb in embeddings]
                dimensions = []
                
                for emb in embeddings:
                    if emb and len(emb) > 0:
                        dimensions.append(len(emb[0]) if emb[0] else 0)
                
                if dimensions:
                    unique_dims = set(dimensions)
                    if len(unique_dims) > 1:
                        issues.append(f"Inconsistent dimensions in '{col_name}': {unique_dims}")
                        health_score -= 20
                    print(f"     ✅ Dimensions: {max(unique_dims) if unique_dims else 0}D (consistent: {len(unique_dims) == 1})")
                    
                    avg_vectors = np.mean(vector_counts)
                    print(f"     ✅ Avg vectors per doc: {avg_vectors:.1f}")
                    
                    if avg_vectors < 5:
                        warnings.append(f"Low vector count per document in '{col_name}' (avg: {avg_vectors:.1f})")
                        health_score -= 5
            
            else:
                # Dense vector specific checks
                dimensions = [len(emb) if emb else 0 for emb in embeddings]
                unique_dims = set(dimensions)
                
                if len(unique_dims) > 1:
                    issues.append(f"Inconsistent dimensions in '{col_name}': {unique_dims}")
                    health_score -= 20
                print(f"     ✅ Dimensions: {max(unique_dims) if unique_dims else 0}D (consistent: {len(unique_dims) == 1})")
        
        # 4. Data consistency checks
        print(f"\n🔍 Data Consistency:")
        
        # Check for duplicate IDs if ID column exists
        id_columns = [col for col in df.columns if 'id' in col.lower()]
        for id_col in id_columns:
            duplicates = df[id_col].duplicated().sum()
            if duplicates > 0:
                issues.append(f"Duplicate IDs in '{id_col}': {duplicates} duplicates")
                health_score -= 15
            print(f"   ✅ {id_col} duplicates: {duplicates}")
        
    except Exception as e:
        issues.append(f"Health check failed: {str(e)}")
        health_score -= 30
    
    # Final health report
    print(f"\n🏆 Health Score: {max(0, health_score)}/100")
    
    if health_score >= 90:
        print(f"   🟢 Excellent - Table is in great condition")
    elif health_score >= 70:
        print(f"   🟡 Good - Minor issues detected")
    elif health_score >= 50:
        print(f"   🟠 Fair - Several issues need attention")
    else:
        print(f"   🔴 Poor - Significant issues detected")
    
    if issues:
        print(f"\n❌ Issues Found:")
        for issue in issues:
            print(f"   • {issue}")
    
    if warnings:
        print(f"\n⚠️  Warnings:")
        for warning in warnings:
            print(f"   • {warning}")
    
    if not issues and not warnings:
        print(f"\n✅ No issues detected - table looks healthy!")

# Run health checks
run_health_checks()

🏥 Health Check: dense_reviews
📊 Basic Checks:
   ✅ Row count: 12
   ✅ Null values: 0 total

🎯 Embedding Checks:

   📋 dense_embedding (Dense):

🏆 Health Score: 70/100
   🟡 Good - Minor issues detected

❌ Issues Found:
   • Health check failed: The truth value of an array with more than one element is ambiguous. Use a.any() or a.all()


## 8. Export & Reporting

In [ ]:
def generate_comprehensive_report():
    """Generate a comprehensive report of all tables"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    report = {
        "report_info": {
            "generated_at": datetime.now().isoformat(),
            "vector_store_path": explorer.vector_store_path,
            "total_tables": len(explorer.tables_info)
        },
        "tables": {}
    }
    
    print(f"📊 Generating Comprehensive Report...")
    print("=" * 60)
    
    for table_name, info in explorer.tables_info.items():
        try:
            # Get basic table info
            df = info['table'].to_pandas()
            
            table_report = {
                "basic_info": {
                    "row_count": len(df),
                    "column_count": len(info['schema']),
                    "table_type": explorer._guess_table_type(table_name, info),
                    "memory_usage_mb": df.memory_usage(deep=True).sum() / 1024 / 1024
                },
                "schema": {
                    "columns": [{
                        "name": field.name,
                        "type": str(field.type),
                        "is_embedding": any(emb['name'] == field.name for emb in info['embedding_columns'])
                    } for field in info['schema']]
                },
                "embeddings": []
            }
            
            # Analyze embeddings
            for emb_col in info['embedding_columns']:
                col_name = emb_col['name']
                embeddings = df[col_name].tolist()
                
                if emb_col['is_multi_vector']:
                    vector_counts = [len(emb) if emb else 0 for emb in embeddings]
                    dimensions = [len(emb[0]) if emb and emb[0] else 0 for emb in embeddings]
                    
                    emb_analysis = {
                        "column_name": col_name,
                        "type": "multi_vector",
                        "dimensions": max(dimensions) if dimensions else 0,
                        "total_vectors": sum(vector_counts),
                        "avg_vectors_per_doc": np.mean(vector_counts) if vector_counts else 0,
                        "min_vectors_per_doc": min(vector_counts) if vector_counts else 0,
                        "max_vectors_per_doc": max(vector_counts) if vector_counts else 0
                    }
                else:
                    dimensions = [len(emb) if emb else 0 for emb in embeddings]
                    
                    emb_analysis = {
                        "column_name": col_name,
                        "type": "dense_vector",
                        "dimensions": max(dimensions) if dimensions else 0,
                        "total_vectors": len(embeddings),
                        "consistent_dimensions": len(set(dimensions)) <= 1
                    }
                
                table_report["embeddings"].append(emb_analysis)
            
            report["tables"][table_name] = table_report
            print(f"   ✅ {table_name}: {len(df):,} rows, {len(info['schema'])} columns")
            
        except Exception as e:
            print(f"   ❌ {table_name}: Failed to analyze - {e}")
            report["tables"][table_name] = {"error": str(e)}
    
    # Save report
    report_path = f"vector_store_report_{timestamp}.json"
    with open(report_path, 'w') as f:
        json.dump(report, f, indent=2, default=str)
    
    print(f"\n💾 Report saved to: {report_path}")
    
    # Display summary
    print(f"\n📋 Summary:")
    total_rows = sum(info.get('basic_info', {}).get('row_count', 0) for info in report['tables'].values())
    total_tables = len([t for t in report['tables'].values() if 'error' not in t])
    
    print(f"   Tables analyzed: {total_tables}/{len(explorer.tables_info)}")
    print(f"   Total rows: {total_rows:,}")
    
    # Show table types
    table_types = {}
    for table_info in report['tables'].values():
        if 'basic_info' in table_info:
            table_type = table_info['basic_info']['table_type']
            table_types[table_type] = table_types.get(table_type, 0) + 1
    
    if table_types:
        print(f"   Table types: {dict(table_types)}")
    
    return report_path

def export_table_sample(table_name: str = None, num_rows: int = 10):
    """Export sample data from current or specified table"""
    target_table = table_name or explorer.current_table_name
    
    if not target_table or target_table not in explorer.tables_info:
        print(f"❌ Table '{target_table}' not found")
        return
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    try:
        table = explorer.tables_info[target_table]['table']
        sample_df = table.head(num_rows).to_pandas()
        
        # Remove embedding columns for readability
        embedding_cols = [col['name'] for col in explorer.tables_info[target_table]['embedding_columns']]
        display_df = sample_df.drop(columns=embedding_cols, errors='ignore')
        
        # Save to CSV
        csv_path = f"{target_table}_sample_{timestamp}.csv"
        display_df.to_csv(csv_path, index=False)
        
        print(f"📊 Sample from '{target_table}' (first {num_rows} rows):")
        display(display_df)
        
        print(f"\n💾 Sample data exported to: {csv_path}")
        print(f"   Note: Embedding columns excluded for readability")
        
        return csv_path
        
    except Exception as e:
        print(f"❌ Export failed: {e}")
        return None

# Generate comprehensive report
report_file = generate_comprehensive_report()

print("\n" + "="*60)

# Export sample from current table
if explorer.current_table_name:
    sample_file = export_table_sample(num_rows=5)
else:
    print("💡 Select a table above to export sample data")

## 🎯 Usage Instructions

### Quick Start:
1. **Run all cells** to initialize the explorer
2. **Select a table** using the dropdown in section 3
3. **Explore data** using the analysis functions
4. **Run health checks** to validate data integrity
5. **Export reports** for documentation

### Advanced Usage:
- **Multi-table comparison**: Compare schemas and statistics across tables
- **Embedding analysis**: Deep dive into vector dimensions and distributions
- **Health monitoring**: Regular validation of data quality
- **Debugging**: Identify issues with vector storage and preprocessing

### Supported Vector Types:
- ✅ **Dense Embeddings**: Single vector per document
- ✅ **Multi-Vector (ColBERT)**: Multiple vectors per document
- ✅ **Mixed Tables**: Tables with multiple embedding columns

---
*This diagnostic tool is designed to work across all vector database projects in this repository.*